# Notebook 03 - Agent Tools

**Project:** IncidentIQ - AI-powered Incident Intelligence

**Goal:** Build all LangChain tools that the LangGraph agent will use in Notebook 04.

## What this notebook covers
1. Install and import all required packages
2. Tool 1 - YouTube transcript tool with Pinecone caching
3. Tool 2 - RAG knowledge search with query rewriting and multi-query
4. Tool 3 - Video summarization tool
5. Tool 4 - PDF cheatsheet generator tool
6. Tool 5 - Gmail sender tool
7. Collect all tools
8. Run quality tests

---

## 1. Install required packages

In [1]:
!pip install langchain langchain-openai langchain-community langchain-text-splitters langchain-pinecone pinecone youtube-transcript-api reportlab google-api-python-client google-auth-oauthlib python-dotenv -q
print('Packages installed.')


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Packages installed.


## 2. Import libraries and load environment variables

In [2]:
import os
import re
import base64
import json
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

from langchain.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pinecone import Pinecone

from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib.colors import HexColor, white
from reportlab.pdfgen import canvas

from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders

load_dotenv()
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY not found.'
assert os.getenv('PINECONE_API_KEY'), 'PINECONE_API_KEY not found.'
print('Environment variables loaded successfully.')

Environment variables loaded successfully.


## 3. Configuration

In [3]:
INDEX_NAME        = 'incidentiq'
EMBEDDING_MODEL   = 'text-embedding-3-small'
LLM_MODEL         = 'gpt-4o-mini'
RETRIEVER_K       = 8
DISTRIBUTION_LIST = os.getenv('GMAIL_DISTRIBUTION_LIST', '').split(',')
GMAIL_SCOPES      = ['https://www.googleapis.com/auth/gmail.send']

RED    = HexColor('#C0392B')
DARK   = HexColor('#1C2833')
ORANGE = HexColor('#E67E22')
GREEN  = HexColor('#1E8449')
WHITE  = white

llm             = ChatOpenAI(model=LLM_MODEL, temperature=0)
embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL)
pc              = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
vectorstore     = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embedding_model,
    pinecone_api_key=os.getenv('PINECONE_API_KEY'),
)

print('Configuration set.')
print(f'   LLM:            {LLM_MODEL}')
print(f'   Pinecone index: {INDEX_NAME}')

Configuration set.
   LLM:            gpt-4o-mini
   Pinecone index: incidentiq


## 4. Tool 1 - YouTube transcript tool

This tool fetches the transcript and stores it in Pinecone cloud.
Key feature: Pinecone cache check — YouTube is only called once per video ever.
Subsequent loads use the cached data from Pinecone — no IP blocking risk.

In [4]:
def extract_video_id(url):
    if 'v=' in url: return url.split('v=')[1].split('&')[0]
    elif 'youtu.be/' in url: return url.split('youtu.be/')[1].split('?')[0]
    raise ValueError(f'Cannot extract video ID: {url}')

def clean_transcript(text):
    text = re.sub(r'\[Music\]|\[Applause\]|\[Laughter\]|\[Cheering\]', '', text)
    text = re.sub(r'\[\d{2}:\d{2}\]\s*(?=\[\d{2}:\d{2}\])', '', text)
    return re.sub(r'\s+', ' ', text).strip()

@tool
def fetch_youtube_transcript(youtube_url: str) -> str:
    """
    Fetch the transcript of a YouTube video and store it in Pinecone cloud.
    Use this tool when the user provides a YouTube URL.
    Checks Pinecone cache first - YouTube is only called once per video ever.
    Supports English, Dutch and French transcripts.
    Returns a confirmation with the number of chunks stored.
    """
    try:
        video_id = extract_video_id(youtube_url)

        # Check Pinecone cache first - never fetch same video twice
        index = pc.Index(INDEX_NAME)
        stats = index.describe_index_stats()
        if stats.total_vector_count > 0:
            # Check if this specific video is already stored
            test = vectorstore.similarity_search(
                'incident', k=1,
                filter={'video_id': video_id} if hasattr(vectorstore, 'filter') else None
            )
            if test:
                return (
                    f'Video already in knowledge base.\n'
                    f'Video ID: {video_id}\n'
                    f'Using cached data from Pinecone - no YouTube request needed.\n'
                    f'Ready for Q&A.'
                )

        # Fetch from YouTube - only if not cached
        try:
            entries = YouTubeTranscriptApi().fetch(video_id, languages=['en', 'nl', 'fr'])
            transcript_list = entries.snippets
        except NoTranscriptFound:
            return f'No transcript found for video {video_id}. Please use a video with CC subtitles.'
        except TranscriptsDisabled:
            return f'Transcripts are disabled for video {video_id}.'
        except Exception as e:
            return (
                f'Could not fetch transcript.\n'
                f'If YouTube is blocking your IP, wait 30-60 minutes and try again.\n'
                f'Error: {str(e)}'
            )

        plain = ' '.join(t.text for t in transcript_list)
        timestamped = ' '.join(
            f'[{int(t.start//60):02d}:{int(t.start%60):02d}] {t.text}'
            for t in transcript_list
        )
        plain       = clean_transcript(plain)
        timestamped = clean_transcript(timestamped)

        splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        chunks   = splitter.create_documents(
            texts=[timestamped],
            metadatas=[{'video_id': video_id, 'source': youtube_url}]
        )
        vectorstore.add_documents(chunks)

        return (
            f'Transcript loaded successfully.\n'
            f'Video ID: {video_id}\n'
            f'Characters: {len(plain):,}\n'
            f'Chunks stored in Pinecone: {len(chunks)}\n'
            f'Ready for Q&A.'
        )
    except Exception as e:
        return f'Error: {str(e)}'

print('Tool 1 defined - fetch_youtube_transcript')

Tool 1 defined - fetch_youtube_transcript


## 5. Tool 2 - RAG knowledge search with query rewriting and multi-query

This tool searches Pinecone for relevant information.
Combines query rewriting and multi-query for maximum retrieval quality.

Query rewriting: makes vague queries more specific before searching.
Multi-query: generates 3 variations and retrieves chunks for each.
Result: more relevant chunks found, better answers generated.

In [5]:
@tool
def search_video_knowledge(query: str) -> str:
    """
    Search the Pinecone knowledge base for information relevant to the query.
    Use this tool to answer questions about the loaded video content.
    Uses query rewriting and multi-query for maximum retrieval quality.
    Automatically translates non-English queries for better results.
    Returns relevant transcript excerpts with timestamp sources.
    """
    try:
        # Step 1 - Translate to English
        english_query = llm.invoke(
            f'Translate to English, return only the translation: {query}'
        ).content.strip()

        # Step 2 - Rewrite for better retrieval
        rewritten = llm.invoke(
            f'Rewrite this search query to be more specific for searching '
            f'an incident training video transcript. '
            f'Return only the rewritten query, maximum 20 words.\n\n'
            f'Query: {english_query}\nRewritten:'
        ).content.strip()

        # Step 3 - Generate multi-query variations
        try:
            response = llm.invoke(
                f'Generate 3 different versions of this question for retrieval. '
                f'Return only a JSON list of 3 strings.\n\n'
                f'Question: {rewritten}\nJSON:'
            ).content.strip()
            raw     = re.sub(r'```json|```', '', response).strip()
            queries = json.loads(raw)
        except Exception:
            queries = [rewritten]
        queries.append(rewritten)

        # Step 4 - Retrieve and deduplicate
        all_docs = {}
        for q in queries:
            results = vectorstore.similarity_search(q, k=4)
            for doc in results:
                key = doc.page_content[:100]
                if key not in all_docs:
                    all_docs[key] = doc

        if not all_docs:
            return 'No relevant information found. Please load a YouTube video first.'

        combined = list(all_docs.values())

        # Step 5 - Extract timestamps
        all_ts = re.findall(r'\[\d{2}:\d{2}\]', ' '.join([d.page_content for d in combined]))
        seen, unique_ts = set(), []
        for t in all_ts:
            if t not in seen:
                seen.add(t)
                unique_ts.append(t)

        clean_chunks = [
            re.sub(r'\[\d{2}:\d{2}\]\s*(?=\[\d{2}:\d{2}\])', '', d.page_content)
            for d in combined
        ]

        return (
            '\n\n'.join(clean_chunks) +
            f'\n\nSources: {" | ".join(unique_ts[:5])}'
            f'\nChunks retrieved: {len(clean_chunks)} via {len(queries)} query variations'
        )
    except Exception as e:
        return f'Error searching knowledge base: {str(e)}'

print('Tool 2 defined - search_video_knowledge')

Tool 2 defined - search_video_knowledge


## 6. Tool 3 - Video summarization tool

In [6]:
@tool
def summarize_video(language: str = 'english') -> str:
    """
    Generate a structured summary of the entire loaded video.
    Use this tool when the user asks for a full summary or overview.
    Specify language as english, dutch or french.
    Returns a structured summary with introduction, key points, lessons and conclusion.
    """
    try:
        results = vectorstore.similarity_search(
            'main topic lessons learned conclusions key points', k=12
        )
        if not results:
            return 'No video content found. Please load a YouTube video first.'

        context = '\n\n'.join([
            re.sub(r'\[\d{2}:\d{2}\]\s*(?=\[\d{2}:\d{2}\])', '', r.page_content)
            for r in results
        ])
        lang_map = {
            'english': 'English',
            'dutch':   'Dutch - use natural direct language as used in Belgian incident training',
            'french':  'French',
        }
        lang = lang_map.get(language.lower(), 'English')
        prompt = (
            f'You are an incident training expert. Write a structured summary in {lang}.\n'
            f'Structure: **Introduction** / **Key Points** / **Lessons Learned** / **Conclusion**\n'
            f'Rules: bullet points only, max 15 words per bullet, based strictly on context.\n\n'
            f'Context:\n{context}\n\nSummary:'
        )
        return llm.invoke(prompt).content.strip()
    except Exception as e:
        return f'Error generating summary: {str(e)}'

print('Tool 3 defined - summarize_video')

Tool 3 defined - summarize_video


## 7. Tool 4 - PDF cheatsheet generator tool

In [7]:
def extract_keypoints_for_pdf(context, language='dutch'):
    lang_map = {'dutch': 'Dutch', 'english': 'English', 'french': 'French'}
    lang = lang_map.get(language.lower(), 'Dutch')
    prompt = (
        f'Extract structured information for an incident training cheatsheet.\n'
        f'Respond in {lang} with this exact JSON and nothing else:\n'
        f'{{"title": "Short title", "subtitle": "Source name", '
        f'"keypoints": ["point 1", "point 2", "point 3", "point 4", "point 5"], '
        f'"recommendations": ["rec 1", "rec 2", "rec 3", "rec 4"]}}\n\n'
        f'Rules: max 12 words per item, no jargon, no timestamps.\n\nContext:\n{context}\n\nJSON:'
    )
    response = llm.invoke(prompt)
    raw = re.sub(r'```json|```', '', response.content.strip()).strip()
    return json.loads(raw)

def generate_pdf(data, source_url=''):
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filepath  = f'/tmp/incidentiq_cheatsheet_{timestamp}.pdf'
    c = canvas.Canvas(filepath, pagesize=A4)
    W, H = A4
    c.setFillColor(RED)
    c.rect(0, H-3.2*cm, W, 3.2*cm, fill=1, stroke=0)
    c.setFillColor(WHITE)
    c.circle(1.8*cm, H-1.6*cm, 0.85*cm, fill=1, stroke=0)
    c.setFillColor(RED)
    c.setFont('Helvetica-Bold', 14)
    c.drawCentredString(1.8*cm, H-1.95*cm, 'IQ')
    c.setFillColor(WHITE)
    c.setFont('Helvetica-Bold', 15)
    c.drawString(3.2*cm, H-1.3*cm, data.get('title', 'IncidentIQ Cheatsheet'))
    c.setFont('Helvetica', 10)
    c.drawString(3.2*cm, H-1.85*cm, data.get('subtitle', ''))
    c.setFont('Helvetica', 8)
    c.drawRightString(W-1.2*cm, H-1.3*cm, datetime.now().strftime('%d/%m/%Y'))
    c.drawRightString(W-1.2*cm, H-1.75*cm, 'Generated by IncidentIQ AI')
    c.setFillColor(ORANGE)
    c.rect(0, H-3.6*cm, W, 0.4*cm, fill=1, stroke=0)
    y = H-5.0*cm
    def sh(y, title, color=DARK):
        c.setFillColor(color)
        c.setFont('Helvetica-Bold', 11)
        c.drawString(1.2*cm, y, title.upper())
        c.setStrokeColor(color)
        c.setLineWidth(1.5)
        c.line(1.2*cm, y-0.2*cm, W-1.2*cm, y-0.2*cm)
        return y-0.7*cm
    def bi(y, text, color=DARK, bc=RED):
        c.setFillColor(bc)
        c.circle(1.2*cm, y+0.2*cm, 0.1*cm, fill=1, stroke=0)
        c.setFillColor(color)
        c.setFont('Helvetica', 9.5)
        mw = W-1.5*cm-1.2*cm
        words = text.split()
        line, lines = '', []
        for w in words:
            t = line+w+' '
            if c.stringWidth(t, 'Helvetica', 9.5) < mw:
                line = t
            else:
                lines.append(line.strip())
                line = w+' '
        lines.append(line.strip())
        for i, l in enumerate(lines):
            c.drawString(1.5*cm, y-i*0.45*cm, l)
        return y-len(lines)*0.45*cm-0.3*cm
    y = sh(y, 'Key Points', RED)
    for kp in data.get('keypoints', []):
        y = bi(y, kp)
    y -= 0.4*cm
    y = sh(y, 'AI-generated recommendations based on the video', GREEN)
    c.setFillColor(HexColor('#EAFAF1'))
    c.setStrokeColor(GREEN)
    c.setLineWidth(0.8)
    c.roundRect(1.2*cm, y-0.75*cm, W-2.4*cm, 0.65*cm, 3, fill=1, stroke=1)
    c.setFillColor(GREEN)
    c.setFont('Helvetica-Bold', 8)
    c.drawString(1.55*cm, y-0.3*cm, 'AI analysis:')
    c.setFillColor(DARK)
    c.setFont('Helvetica-Oblique', 8)
    c.drawString(3.0*cm, y-0.3*cm, 'Automatically extracted from video - not cited from a person.')
    y -= 1.0*cm
    for rec in data.get('recommendations', []):
        y = bi(y, rec, bc=GREEN)
    c.setFillColor(RED)
    c.rect(0, 1.2*cm, W, 0.15*cm, fill=1, stroke=0)
    c.setFillColor(DARK)
    c.rect(0, 0, W, 1.2*cm, fill=1, stroke=0)
    c.setFillColor(WHITE)
    c.setFont('Helvetica', 7.5)
    c.drawString(1.2*cm, 0.65*cm, 'IncidentIQ - AI-powered Incident Intelligence')
    if source_url:
        c.drawCentredString(W/2, 0.65*cm, f'Source: {source_url[:70]}')
    c.drawRightString(W-1.2*cm, 0.65*cm, 'Page 1/1')
    c.save()
    return filepath

@tool
def generate_pdf_cheatsheet(language: str = 'dutch', source_url: str = '') -> str:
    """
    Generate a professional 1-page PDF cheatsheet from the loaded video content.
    Use this tool when the user asks for a cheatsheet, key concepts document or PDF.
    Specify language as dutch, english or french.
    Returns the file path of the generated PDF for the Gmail tool to attach.
    """
    try:
        results = vectorstore.similarity_search(
            'key points lessons learned recommendations conclusions', k=10
        )
        if not results:
            return 'No video content found. Please load a YouTube video first.'
        context  = '\n\n'.join([r.page_content for r in results])
        data     = extract_keypoints_for_pdf(context, language)
        filepath = generate_pdf(data, source_url)
        return (
            f'PDF cheatsheet generated successfully.\n'
            f'File path: {filepath}\n'
            f'Title: {data.get("title", "N/A")}'
        )
    except Exception as e:
        return f'Error generating PDF: {str(e)}'

print('Tool 4 defined - generate_pdf_cheatsheet')

Tool 4 defined - generate_pdf_cheatsheet


## 8. Tool 5 - Gmail sender tool

Sends the PDF to the distribution list and any custom email addresses.
custom_emails parameter allows users to type their own address in Streamlit.

In [8]:
def get_gmail_service():
    creds = None
    token_path = Path('../token.json')
    creds_path = Path('../credentials.json')
    assert creds_path.exists(), 'credentials.json not found in project root.'
    if token_path.exists():
        creds = Credentials.from_authorized_user_file(str(token_path), GMAIL_SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(str(creds_path), GMAIL_SCOPES)
            creds = flow.run_local_server(port=0)
        token_path.write_text(creds.to_json())
    return build('gmail', 'v1', credentials=creds)

def create_email_with_attachment(sender, recipients, subject, body, pdf_path):
    msg = MIMEMultipart()
    msg['From']    = sender
    msg['To']      = ', '.join(recipients)
    msg['Subject'] = subject
    msg.attach(MIMEText(body, 'plain'))
    with open(pdf_path, 'rb') as f:
        part = MIMEBase('application', 'octet-stream')
        part.set_payload(f.read())
    encoders.encode_base64(part)
    part.add_header('Content-Disposition', f'attachment; filename={Path(pdf_path).name}')
    msg.attach(part)
    return {'raw': base64.urlsafe_b64encode(msg.as_bytes()).decode()}

@tool
def send_gmail_with_cheatsheet(pdf_path: str, custom_emails: str = '') -> str:
    """
    Send the generated PDF cheatsheet to the distribution list via Gmail.
    Use this tool after generate_pdf_cheatsheet has created the PDF file.
    pdf_path: file path returned by the generate_pdf_cheatsheet tool.
    custom_emails: optional comma-separated emails added to the send list.
    Returns a confirmation with the full list of recipients.
    """
    try:
        assert Path(pdf_path).exists(), f'PDF not found: {pdf_path}'
        recipients = [e.strip() for e in DISTRIBUTION_LIST if e.strip()]
        if custom_emails:
            recipients.extend([e.strip() for e in custom_emails.split(',') if e.strip()])
        assert recipients, 'No recipients. Add GMAIL_DISTRIBUTION_LIST to .env.'
        service = get_gmail_service()
        subject = f'IncidentIQ - Key Concepts Cheatsheet - {datetime.now().strftime("%d/%m/%Y")}'
        body = (
            'Dear colleague,\n\n'
            'Please find attached the AI-generated key concepts cheatsheet.\n'
            'Recommendations should be reviewed by a qualified officer before operational use.\n\n'
            'Best regards,\nIncidentIQ AI Agent'
        )
        message = create_email_with_attachment('me', recipients, subject, body, pdf_path)
        service.users().messages().send(userId='me', body=message).execute()
        return (
            f'Cheatsheet sent successfully.\n'
            f'Recipients: {", ".join(recipients)}'
        )
    except Exception as e:
        return f'Error sending email: {str(e)}'

print('Tool 5 defined - send_gmail_with_cheatsheet')

Tool 5 defined - send_gmail_with_cheatsheet


## 9. Tool 6 - XVR Scenario Creator

In [9]:
@tool
def generate_xvr_scenario(language: str = 'dutch') -> str:
    """
    Generate a structured XVR simulation scenario brief
    based on the loaded incident video.
    Use this tool when the user asks to create an XVR scenario
    or clicks the Create XVR Scenario button.
    Returns a formatted scenario brief for XVR operators.
    """
    try:
        results = vectorstore.similarity_search(
            'location building fire cause complications decisions '
            'resources weather time casualties evacuation', k=12
        )
        if not results:
            return 'No video content found. Please load a YouTube video first.'

        context = '\n\n'.join([
            re.sub(r'\[\d{2}:\d{2}\]\s*(?=\[\d{2}:\d{2}\])', '', r.page_content)
            for r in results
        ])

        lang_map = {
            'dutch':   'Dutch - professional Belgian fire service terminology',
            'english': 'English',
            'french':  'French',
        }
        lang = lang_map.get(language.lower(), 'Dutch')

        prompt = (
            f'You are an expert XVR simulation scenario designer for emergency services.\n'
            f'Based on the incident described in the context, generate a complete\n'
            f'XVR operator scenario brief in {lang}.\n\n'
            f'Use this exact structure:\n\n'
            f'SCENARIO BRIEF — XVR SIMULATION\n'
            f'================================\n\n'
            f'INCIDENT TITLE:\n'
            f'[Short descriptive title]\n\n'
            f'LOCATION & BUILDING:\n'
            f'- Building type: [type]\n'
            f'- Floors: [number]\n'
            f'- Construction: [facade, materials]\n'
            f'- Address/area: [if mentioned]\n\n'
            f'INITIAL SITUATION T+00:00:\n'
            f'- Fire location: [exact location]\n'
            f'- Visibility: [smoke, flames]\n'
            f'- Known casualties: [number and location]\n'
            f'- First resources: [vehicles, personnel]\n\n'
            f'ENVIRONMENTAL CONDITIONS:\n'
            f'- Time of day: [if mentioned]\n'
            f'- Weather: [if mentioned]\n'
            f'- Special hazards: [materials, access]\n\n'
            f'SCENARIO COMPLICATIONS (inject in order):\n'
            f'- T+[time]: [complication 1]\n'
            f'- T+[time]: [complication 2]\n'
            f'- T+[time]: [complication 3]\n'
            f'- T+[time]: [complication 4]\n\n'
            f'CRITICAL DECISION MOMENTS:\n'
            f'1. [Decision moment with context]\n'
            f'2. [Decision moment with context]\n'
            f'3. [Decision moment with context]\n\n'
            f'LEARNING OBJECTIVES:\n'
            f'- [Objective 1]\n'
            f'- [Objective 2]\n'
            f'- [Objective 3]\n\n'
            f'DEBRIEFING QUESTIONS:\n'
            f'1. [Question based on actual mistakes]\n'
            f'2. [Question based on actual mistakes]\n'
            f'3. [Question based on actual mistakes]\n\n'
            f'XVR OPERATOR NOTES:\n'
            f'[Specific notes about key moments to inject]\n\n'
            f'Rules:\n'
            f'- Base everything strictly on the context\n'
            f'- Use realistic timings from the video\n'
            f'- Focus on what actually went wrong\n'
            f'- Never invent details not in the context\n\n'
            f'Context:\n{context}\n\nScenario brief:'
        )

        return llm.invoke(prompt).content.strip()

    except Exception as e:
        return f'Error generating XVR scenario: {str(e)}'


# Test Tool 6
print('Testing Tool 6 - generate_xvr_scenario...')
result = generate_xvr_scenario.invoke({'language': 'dutch'})
print(result)

Testing Tool 6 - generate_xvr_scenario...
SCENARIO BRIEF — XVR SIMULATION

INCIDENT TITLE:
Brand in een hoog gebouw

LOCATION & BUILDING:
- Building type: Hoogbouw
- Floors: 11
- Construction: Oude constructie, enkelvoudige trap
- Address/area: Niet vermeld

INITIAL SITUATION T+00:00:
- Fire location: 11e verdieping
- Visibility: Volledig ontwikkeld vuur, rook
- Known casualties: Onbekend, meldingen van ouderen in nood
- First resources: 1 brandweerwagen, 1 ladderwagen, 1 commandowagen, 1 ambulance

ENVIRONMENTAL CONDITIONS:
- Time of day: Niet vermeld
- Weather: Niet vermeld
- Special hazards: Oud gebouw met beperkte brandpreventie, enkelvoudige trap, risico op vallend glas

SCENARIO COMPLICATIONS (inject in order):
- T+02:00: Eerste eenheid arriveert en constateert dat het vuur op de 11e verdieping volledig ontwikkeld is.
- T+05:00: Rookontwikkeling op de begane grond, meldingen van mensen die hulp nodig hebben.
- T+08:00: Eerste team raakt gedesoriënteerd in de rook en verliest het 

## 9. Collect all tools

In [12]:
AGENT_TOOLS = [
    fetch_youtube_transcript,
    search_video_knowledge,
    summarize_video,
    generate_pdf_cheatsheet,
    send_gmail_with_cheatsheet,
    generate_xvr_scenario,      
]

print('All tools collected.')
print(f'   Total: {len(AGENT_TOOLS)} tools')
for i, t in enumerate(AGENT_TOOLS, 1):
    print(f'   Tool {i}: {t.name}')

All tools collected.
   Total: 6 tools
   Tool 1: fetch_youtube_transcript
   Tool 2: search_video_knowledge
   Tool 3: summarize_video
   Tool 4: generate_pdf_cheatsheet
   Tool 5: send_gmail_with_cheatsheet
   Tool 6: generate_xvr_scenario


## 10. Tool quality tests

In [15]:
print('=' * 60)
print('TOOL QUALITY TESTS')
print('=' * 60)

tests_passed = 0
tests_failed = 0

def check(name, condition, detail=''):
    global tests_passed, tests_failed
    if condition:
        tests_passed += 1
        print(f'  PASS - {name}')
    else:
        tests_failed += 1
        print(f'  FAIL - {name}')
    if detail:
        print(f'         {detail}')

# Test 1 - Tool 1 detects cached video
r = fetch_youtube_transcript.invoke({'youtube_url': 'https://www.youtube.com/watch?v=7OH5FEWWM_k'})
check('Tool 1 - handles YouTube URL (cached or fetched)',
    'Ready for Q&A' in r or 'Chunks stored' in r or 'cached' in r.lower(),
    f'Result: {r[:80]}...')

# Test 2 - Tool 1 handles invalid URL
r = fetch_youtube_transcript.invoke({'youtube_url': 'https://not-youtube.com'})
check('Tool 1 - handles invalid URL gracefully',
    'Error' in r or 'Cannot' in r,
    f'Result: {r[:80]}')

# Test 3 - Tool 2 returns context with multi-query
r = search_video_knowledge.invoke({'query': 'What went wrong during the incident?'})
check('Tool 2 - returns relevant context with multi-query',
    len(r) > 100 and 'Sources' in r,
    f'Length: {len(r)} chars')

# Test 4 - Tool 2 handles Dutch query
r = search_video_knowledge.invoke({'query': 'Welke lessen werden geleerd?'})
check('Tool 2 - handles Dutch query via translation and rewriting',
    len(r) > 100,
    f'Length: {len(r)} chars')

# Test 5 - Tool 3 returns structured summary
r = summarize_video.invoke({'language': 'dutch'})
check('Tool 3 - returns structured Dutch summary',
    len(r) > 200 and ('**' in r or '-' in r),
    f'Length: {len(r)} chars')

# Test 6 - Tool 4 generates PDF
r = generate_pdf_cheatsheet.invoke({'language': 'dutch', 'source_url': 'https://www.youtube.com/watch?v=7OH5FEWWM_k'})
pdf_ok = 'File path:' in r and Path(r.split('File path: ')[1].split('\n')[0]).exists()
check('Tool 4 - generates PDF on disk',
    pdf_ok,
    f'Result: {r[:80]}...')

# Test 7 - Tool 5 callable
check('Tool 5 - send_gmail_with_cheatsheet is defined',
    send_gmail_with_cheatsheet.name == 'send_gmail_with_cheatsheet',
    'Gmail requires credentials.json for actual sending - tested during deployment')

# Test 8 - Tool 6 generates XVR scenario
r = generate_xvr_scenario.invoke({'language': 'dutch'})
check('Tool 6 - generate_xvr_scenario returns structured brief',
    len(r) > 200 and 'SCENARIO' in r.upper(),
    f'Length: {len(r)} chars')

# Test 9 - Tool 6 works in English
r_en = generate_xvr_scenario.invoke({'language': 'english'})
check('Tool 6 - generate_xvr_scenario works in English',
    len(r_en) > 200 and 'SCENARIO' in r_en.upper(),
    f'Length: {len(r_en)} chars')

# Test 10 - All 6 tools collected
check('All 6 tools in AGENT_TOOLS list',
    len(AGENT_TOOLS) == 6,
    f'Tools: {len(AGENT_TOOLS)}')

print('\n' + '=' * 60)
print(f'RESULTS: {tests_passed} passed | {tests_failed} failed')
if tests_failed == 0:
    print('All tests passed - tools ready for Notebook 04!')
else:
    print('Fix failing tests before moving to Notebook 04.')
print('=' * 60)

TOOL QUALITY TESTS
  PASS - Tool 1 - handles YouTube URL (cached or fetched)
         Result: Video already in knowledge base.
Video ID: 7OH5FEWWM_k
Using cached data from Pi...
  PASS - Tool 1 - handles invalid URL gracefully
         Result: Error: Cannot extract video ID: https://not-youtube.com
  PASS - Tool 2 - returns relevant context with multi-query
         Length: 3095 chars
  PASS - Tool 2 - handles Dutch query via translation and rewriting
         Length: 2593 chars
  PASS - Tool 3 - returns structured Dutch summary
         Length: 970 chars
  PASS - Tool 4 - generates PDF on disk
         Result: PDF cheatsheet generated successfully.
File path: /tmp/incidentiq_cheatsheet_202...
  PASS - Tool 5 - send_gmail_with_cheatsheet is defined
         Gmail requires credentials.json for actual sending - tested during deployment
  PASS - Tool 6 - generate_xvr_scenario returns structured brief
         Length: 2282 chars
  PASS - Tool 6 - generate_xvr_scenario works in English
    

---

## What we built

| Tool | Trigger | What |
|------|---------|------|
| fetch_youtube_transcript | YouTube URL | Pinecone cache check then YouTube |
| search_video_knowledge | Question | Query rewriting + multi-query + Pinecone |
| summarize_video | Summary request | Structured summary any language |
| generate_pdf_cheatsheet | Cheatsheet request | Branded 1-page PDF |
| send_gmail_with_cheatsheet | Email request | PDF to list + custom email |

## Next: Notebook 04 - LangGraph Agent